In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO

In [3]:
clf_path = "/home/vlad/eQTL/sqanti3_output/wtc11_classification.txt"
df = pd.read_csv(clf_path, sep="\t")

print(df.shape)
print(len(df.columns))
print(df.columns.tolist())

(489545, 55)
55
['isoform', 'chrom', 'start', 'end', 'strand', 'length', 'exons', 'structural_category', 'subcategory', 'FSM_class', 'associated_gene', 'associated_transcript', 'ref_length', 'ref_exons', 'diff_to_TSS', 'diff_to_TTS', 'diff_to_TSS_genomic', 'diff_to_TTS_genomic', 'diff_to_gene_TSS', 'diff_to_gene_TTS', 'RTS_stage', 'all_canonical', 'min_sample_cov', 'min_cov', 'min_cov_pos', 'sd_cov', 'FL', 'n_indels', 'n_indels_junc', 'bite', 'iso_exp', 'gene_exp', 'ratio_exp', 'coding', 'ORF_length', 'CDS_length', 'protein_length', 'CDS_start', 'CDS_end', 'CDS_genomic_start', 'CDS_genomic_end', 'psauron_score', 'CDS_type', 'predicted_NMD', 'perc_A_downstream_TTS', 'seq_A_downstream_TTS', 'dist_to_CAGE_peak', 'within_CAGE_peak', 'dist_to_polyA_site', 'within_polyA_site', 'polyA_motif', 'polyA_dist', 'polyA_motif_found', 'ratio_TSS', 'ref_num_exons']


In [4]:
df.head()

,isoform,chrom,start,end,strand,length,exons,structural_category,subcategory,FSM_class,...,seq_A_downstream_TTS,dist_to_CAGE_peak,within_CAGE_peak,dist_to_polyA_site,within_polyA_site,polyA_motif,polyA_dist,polyA_motif_found,ratio_TSS,ref_num_exons
0,ENCLB377SXJT000206762,chr1,14403,29570,-,1773,10,novel_not_in_catalog,at_least_one_novel_splicesite,C,...,CCAACAGTGTGCTTTTAATA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ENCLB377SXJT000206763,chr1,14403,29570,-,2643,8,novel_not_in_catalog,intron_retention,C,...,CCAACAGTGTGCTTTTAATA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ENCLB377SXJT000206764,chr1,14403,29570,-,2290,11,novel_not_in_catalog,at_least_one_novel_splicesite,C,...,CCAACAGTGTGCTTTTAATA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ENCLB377SXJT000206765,chr1,14403,21129,-,4150,8,novel_not_in_catalog,at_least_one_novel_splicesite,C,...,CCAACAGTGTGCTTTTAATA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENCLB377SXJT000206766,chr1,14403,29570,-,2824,9,novel_not_in_catalog,intron_retention,C,...,CCAACAGTGTGCTTTTAATA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
print(df["structural_category"].value_counts())

missing = (df.isna().sum() / len(df) * 100).round(1)
print(missing[missing > 0].sort_values(ascending=False))

structural_category
full-splice_match          204341
novel_not_in_catalog        99242
novel_in_catalog            85493
incomplete-splice_match     67457
genic_intron                12043
antisense                    6639
genic                        5371
intergenic                   4844
fusion                       4115
Name: count, dtype: int64
iso_exp                100.0
n_indels               100.0
FL                     100.0
min_cov_pos            100.0
sd_cov                 100.0
min_sample_cov         100.0
min_cov                100.0
within_CAGE_peak       100.0
dist_to_polyA_site     100.0
within_polyA_site      100.0
CDS_genomic_end        100.0
psauron_score          100.0
CDS_type               100.0
predicted_NMD          100.0
n_indels_junc          100.0
CDS_length             100.0
ORF_length             100.0
gene_exp               100.0
ratio_exp              100.0
protein_length         100.0
CDS_genomic_start      100.0
CDS_end                100.0
CDS_start 

In [6]:
# Какие колонки реально используемы
usable = df.isna().sum()[df.isna().sum() < len(df) * 0.5]
print("=== КОЛОНКИ С < 50% NaN ===")
print(usable.sort_values())

# Посмотрим на реально заполненные числовые фичи
numeric_cols = df.select_dtypes(include=[np.number]).columns
print("\n=== ЧИСЛОВЫЕ ФИЧИ БЕЗ NaN ===")
for col in numeric_cols:
    nan_pct = df[col].isna().mean() * 100
    if nan_pct == 0:
        print(f"  {col}: min={df[col].min():.1f}, max={df[col].max():.1f}, mean={df[col].mean():.1f}")

=== КОЛОНКИ С < 50% NaN ===
isoform                       0
chrom                         0
start                         0
end                           0
strand                        0
length                        0
exons                         0
structural_category           0
subcategory                   0
FSM_class                     0
associated_gene               0
associated_transcript         0
RTS_stage                     0
perc_A_downstream_TTS         0
coding                        0
seq_A_downstream_TTS          1
diff_to_gene_TTS          16887
diff_to_gene_TSS          16887
ref_exons                 26302
all_canonical             58969
bite                      58969
diff_to_TTS              217747
ref_length               217747
diff_to_TSS              217747
diff_to_TTS_genomic      217747
diff_to_TSS_genomic      217747
dtype: int64

=== ЧИСЛОВЫЕ ФИЧИ БЕЗ NaN ===
  start: min=16.0, max=248936580.0, mean=73183867.7
  end: min=577.0, max=248937043.0, mean=7322

In [7]:
print(df["coding"].value_counts(dropna=False))
print("\n")
print(df["coding"].isna().sum())
print(df["coding"].unique())

coding
non_coding    489545
Name: count, dtype: int64


0
['non_coding']


In [8]:
import pandas as pd

# Парсим GFF3 — берём только mRNA строки где есть len и type
orfs = []
with open("/home/vlad/eQTL/td2_output/longest_orfs.gff3") as f:
    for line in f:
        if "\tmRNA\t" not in line:
            continue
        parts = line.strip().split("\t")
        isoform_id = parts[0]
        attrs = parts[8]
        
        # Извлекаем type и len
        import re
        orf_type = re.search(r'type:(\w+)', attrs)
        orf_len = re.search(r'len:(\d+)', attrs)
        
        if orf_type and orf_len:
            orfs.append({
                'isoform': isoform_id,
                'orf_type': orf_type.group(1),
                'protein_length': int(orf_len.group(1))
            })

orf_df = pd.DataFrame(orfs)
print(f"ORF записей: {len(orf_df)}")
print(orf_df['orf_type'].value_counts())

ORF записей: 994870
orf_type
complete          702058
5prime_partial    251104
3prime_partial     29417
internal           12291
Name: count, dtype: int64


In [9]:
# Приоритет типов ORF
type_priority = {
    'complete': 0,
    '5prime_partial': 1, 
    '3prime_partial': 2,
    'internal': 3
}

orf_df['priority'] = orf_df['orf_type'].map(type_priority)

# Для каждой изоформы берём лучший ORF
# Сначала по приоритету типа, потом по длине
best_orf = (orf_df
    .sort_values(['isoform', 'priority', 'protein_length'], 
                 ascending=[True, True, False])
    .groupby('isoform')
    .first()
    .reset_index()
)

print(f"Уникальных изоформ с ORF: {len(best_orf)}")
print(best_orf['orf_type'].value_counts())
print(f"\nДлина белка (аминокислоты):")
print(best_orf['protein_length'].describe())

Уникальных изоформ с ORF: 397430
orf_type
complete          351375
5prime_partial     20606
3prime_partial     15976
internal            9473
Name: count, dtype: int64

Длина белка (аминокислоты):
count    397430.000000
mean        450.925360
std         444.815998
min          90.000000
25%         158.000000
50%         316.000000
75%         589.000000
max       35992.000000
Name: protein_length, dtype: float64


In [10]:
# Мержим с classification
df_merged = df.merge(best_orf[['isoform', 'orf_type', 'protein_length']], 
                     on='isoform', 
                     how='left')

# Заполняем NaN для некодирующих
df_merged['orf_type'] = df_merged['orf_type'].fillna('no_orf')
df_merged['protein_length'] = df_merged['protein_length'].fillna(0)

# Добавляем бинарную фичу coding
df_merged['has_orf'] = (df_merged['orf_type'] != 'no_orf').astype(int)

print(f"Shape после merge: {df_merged.shape}")
print(f"\nhas_orf распределение:")
print(df_merged['has_orf'].value_counts())
print(f"\norf_type по классам:")
print(pd.crosstab(df_merged['structural_category'], 
                  df_merged['orf_type'], 
                  normalize='index').round(2))

KeyError: 'protein_length'

In [11]:
print([c for c in df_merged.columns if 'protein' in c])

['protein_length_x', 'protein_length_y']


In [12]:
# Убираем старую пустую колонку, переименовываем новую
df_merged = df_merged.drop(columns=['protein_length_x'])
df_merged = df_merged.rename(columns={'protein_length_y': 'protein_length'})

# Теперь заполняем
df_merged['orf_type'] = df_merged['orf_type'].fillna('no_orf')
df_merged['protein_length'] = df_merged['protein_length'].fillna(0)
df_merged['has_orf'] = (df_merged['orf_type'] != 'no_orf').astype(int)

print(f"Shape: {df_merged.shape}")
print(f"\nhas_orf:")
print(df_merged['has_orf'].value_counts())
print(f"\norf_type по классам:")
print(pd.crosstab(df_merged['structural_category'], 
                  df_merged['orf_type'], 
                  normalize='index').round(2))

Shape: (489545, 57)

has_orf:
has_orf
1    397430
0     92115
Name: count, dtype: int64

orf_type по классам:
orf_type                 3prime_partial  5prime_partial  complete  internal  \
structural_category                                                           
antisense                          0.00            0.06      0.44      0.00   
full-splice_match                  0.07            0.07      0.51      0.04   
fusion                             0.00            0.02      0.93      0.00   
genic                              0.01            0.06      0.48      0.00   
genic_intron                       0.00            0.02      0.33      0.00   
incomplete-splice_match            0.01            0.04      0.83      0.01   
intergenic                         0.00            0.05      0.41      0.00   
novel_in_catalog                   0.00            0.01      0.96      0.00   
novel_not_in_catalog               0.00            0.01      0.95      0.00   

orf_type            

In [13]:
import os

os.makedirs("/home/vlad/eQTL/processed", exist_ok=True)

out_path = "/home/vlad/eQTL/processed/classification_with_orf.tsv"
df_merged.to_csv(out_path, sep="\t", index=False)